<a href="https://colab.research.google.com/github/Likihth2004/Likihth2004/blob/main/Smart_Document_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🌐 Smart Document Assistant
# Modules:
# 1. PDF Text Extraction      -> PyMuPDF
# 2. Language Detection       -> langdetect
# 3. Text Chunking            -> Python
# 4. Multilingual Embeddings  -> Sentence Transformers
# 5. Vector Search            -> FAISS
# 6. Indic NLP                -> IndicBERT
# 7. Speech-to-Text            -> Whisper
# 8. Document QA               -> Qwen
# 9. Translation               -> NLLB-200
# 10. User Interface           -> Gradio
#
# Languages:
# English / Kannada / Hindi
# ============================================================


# ------------------------------------------------------------
# 1. INSTALL LIBRARIES
# ------------------------------------------------------------

!pip install -q \
    "transformers>=4.51,<5" \
    sentence-transformers \
    faiss-cpu \
    pymupdf \
    langdetect \
    gradio \
    accelerate \
    sentencepiece


# ------------------------------------------------------------
# 2. IMPORTS
# ------------------------------------------------------------

import os
import gc
import numpy as np
import pandas as pd
import torch
import fitz
import faiss
import gradio as gr

from langdetect import detect

from google.colab import userdata

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    pipeline
)

from sentence_transformers import SentenceTransformer


os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 70)
print("🌐 MULTILINGUAL AI DOCUMENT ASSISTANT")
print("=" * 70)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


# ------------------------------------------------------------
# 3. HUGGING FACE TOKEN
# ------------------------------------------------------------

HF_TOKEN = None

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("✅ HF_TOKEN found in Colab Secrets")
except Exception:
    print("⚠️ HF_TOKEN not found in Colab Secrets")


# ------------------------------------------------------------
# 4. LOAD MULTILINGUAL EMBEDDING MODEL
# ------------------------------------------------------------

print("\n⏳ Loading multilingual embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=DEVICE
)

print("✅ Embedding model loaded")


# ------------------------------------------------------------
# 5. LOAD INDICBERT
# ------------------------------------------------------------

print("\n⏳ Loading IndicBERT...")

indicbert_tokenizer = None
indicbert_model = None
INDICBERT_STATUS = ""

try:

    if HF_TOKEN is None:
        raise RuntimeError(
            "HF_TOKEN is required for ai4bharat/indic-bert. "
            "Add HF_TOKEN in Colab Secrets."
        )

    indicbert_tokenizer = AutoTokenizer.from_pretrained(
        "ai4bharat/indic-bert",
        token=HF_TOKEN
    )

    indicbert_model = AutoModel.from_pretrained(
        "ai4bharat/indic-bert",
        token=HF_TOKEN
    ).to(DEVICE)

    indicbert_model.eval()

    INDICBERT_STATUS = "✅ IndicBERT loaded successfully"

    print(INDICBERT_STATUS)

except Exception as e:

    INDICBERT_STATUS = (
        "⚠️ IndicBERT could not be loaded: "
        + str(e)
    )

    print(INDICBERT_STATUS)
    print(
        "The application can still run, but accept "
        "IndicBERT access conditions first."
    )


# ------------------------------------------------------------
# 6. INDICBERT REPRESENTATION FUNCTION
# ------------------------------------------------------------

def indicbert_encode(text):

    if indicbert_tokenizer is None or indicbert_model is None:
        return None

    if not text or not text.strip():
        return None

    inputs = indicbert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(DEVICE)

    with torch.no_grad():

        outputs = indicbert_model(
            **inputs
        )

    hidden = outputs.last_hidden_state

    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    masked_hidden = hidden * attention_mask

    pooled = (
        masked_hidden.sum(dim=1)
        /
        attention_mask.sum(dim=1).clamp(min=1)
    )

    return pooled.cpu().numpy()[0]


# ------------------------------------------------------------
# 7. LOAD QWEN
# ------------------------------------------------------------

print("\n⏳ Loading Qwen...")

qwen_name = "Qwen/Qwen2.5-0.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(
    qwen_name
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    torch_dtype=torch.float32
).to(DEVICE)

qwen_model.eval()

print("✅ Qwen loaded")


# ------------------------------------------------------------
# 8. LOAD NLLB TRANSLATION MODEL
# ------------------------------------------------------------

print("\n⏳ Loading NLLB translation model...")

nllb_name = "facebook/nllb-200-distilled-600M"

nllb_tokenizer = AutoTokenizer.from_pretrained(
    nllb_name
)

nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    nllb_name,
    torch_dtype=torch.float32
).to(DEVICE)

nllb_model.eval()

print("✅ NLLB loaded")


# ------------------------------------------------------------
# 9. LOAD WHISPER
# ------------------------------------------------------------

print("\n⏳ Loading Whisper...")

whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=-1 if DEVICE == "cpu" else 0
)

print("✅ Whisper loaded")


# ------------------------------------------------------------
# 10. LANGUAGE CODES
# ------------------------------------------------------------

LANG_CODES = {
    "English": "eng_Latn",
    "Kannada": "kan_Knda",
    "Hindi": "hin_Deva"
}

DETECT_CODES = {
    "en": "English",
    "kn": "Kannada",
    "hi": "Hindi",
    "ta": "Tamil",
    "te": "Telugu",
    "ml": "Malayalam",
    "mr": "Marathi",
    "bn": "Bengali",
    "gu": "Gujarati"
}


# ------------------------------------------------------------
# 11. LANGUAGE DETECTION
# ------------------------------------------------------------

def detect_language(text):

    if not text or not text.strip():
        return "English"

    try:

        code = detect(text)

        return DETECT_CODES.get(
            code,
            "English"
        )

    except Exception:

        return "English"


# ------------------------------------------------------------
# 12. NLLB TRANSLATION FUNCTION
# ------------------------------------------------------------

def translate_nllb(
    text,
    source_language,
    target_language
):

    if not text or not text.strip():
        return ""

    if source_language == target_language:
        return text

    if source_language not in LANG_CODES:
        return text

    if target_language not in LANG_CODES:
        return text

    source_code = LANG_CODES[source_language]
    target_code = LANG_CODES[target_language]

    # Set source language
    nllb_tokenizer.src_lang = source_code

    # Tokenize
    inputs = nllb_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    # NLLB target language token
    target_token_id = (
        nllb_tokenizer.convert_tokens_to_ids(
            target_code
        )
    )

    with torch.no_grad():

        output_tokens = nllb_model.generate(
            **inputs,
            forced_bos_token_id=target_token_id,
            max_length=512,
            num_beams=4
        )

    translated = nllb_tokenizer.batch_decode(
        output_tokens,
        skip_special_tokens=True
    )[0]

    return translated.strip()


# ------------------------------------------------------------
# 13. SPEECH TO TEXT
# ------------------------------------------------------------

def transcribe_audio(audio_path):

    if audio_path is None:
        return ""

    try:

        result = whisper_pipe(
            audio_path,
            generate_kwargs={
                "task": "transcribe"
            }
        )

        return result["text"].strip()

    except Exception as e:

        return f"[Speech error: {str(e)}]"


# ------------------------------------------------------------
# 14. TEXT CHUNKING
# ------------------------------------------------------------

def split_into_chunks(
    text,
    chunk_size=1000,
    overlap=200
):

    text = " ".join(text.split())

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += (
            chunk_size - overlap
        )

    return chunks


# ------------------------------------------------------------
# 15. DOCUMENT STATE
# ------------------------------------------------------------

current_chunks = []
current_index = None
current_document_language = "English"
current_pdf_path = None


# ------------------------------------------------------------
# 16. PROCESS PDF
# ------------------------------------------------------------

def process_pdf(pdf_path):

    global current_chunks
    global current_index
    global current_document_language
    global current_pdf_path

    if pdf_path is None:

        return "❌ Please upload a PDF."

    try:

        current_pdf_path = pdf_path

        # Open PDF
        doc = fitz.open(pdf_path)

        page_count = len(doc)

        all_text = ""

        for page_number, page in enumerate(doc):

            page_text = page.get_text()

            all_text += (
                f"\n\n--- Page {page_number + 1} ---\n\n"
                + page_text
            )

        doc.close()

        clean_text = " ".join(
            all_text.split()
        )

        if len(clean_text) < 50:

            return (
                "❌ Very little text was extracted.\n\n"
                "This appears to be a scanned/image PDF.\n"
                "OCR is required for this document."
            )

        # Detect document language
        current_document_language = detect_language(
            clean_text
        )

        # Create chunks
        current_chunks = split_into_chunks(
            all_text,
            chunk_size=1000,
            overlap=200
        )

        # Create embeddings
        embeddings = embedding_model.encode(
            current_chunks,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        ).astype("float32")

        # Create FAISS
        dimension = embeddings.shape[1]

        current_index = faiss.IndexFlatIP(
            dimension
        )

        current_index.add(
            embeddings
        )

        # IndicBERT test representation
        indicbert_message = "Not available"

        if indicbert_model is not None:

            sample_text = clean_text[:500]

            representation = indicbert_encode(
                sample_text
            )

            if representation is not None:

                indicbert_message = (
                    f"Representation created "
                    f"({len(representation)} dimensions)"
                )

        # Clean memory
        del embeddings
        gc.collect()

        return (
            "✅ DOCUMENT PROCESSED\n\n"
            f"📄 Pages: {page_count}\n"
            f"📝 Characters: {len(clean_text)}\n"
            f"📚 Chunks: {len(current_chunks)}\n"
            f"🌐 Language: {current_document_language}\n"
            f"🧠 IndicBERT: {indicbert_message}"
        )

    except Exception as e:

        return (
            "❌ PDF PROCESSING ERROR\n\n"
            + str(e)
        )


# ------------------------------------------------------------
# 17. SEARCH DOCUMENT
# ------------------------------------------------------------

def search_document(
    question,
    top_k=3
):

    if current_index is None:
        return []

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    k = min(
        top_k,
        len(current_chunks)
    )

    scores, indices = current_index.search(
        question_embedding,
        k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx < 0:
            continue

        results.append({
            "score": float(score),
            "chunk": current_chunks[idx],
            "index": int(idx)
        })

    return results


# ------------------------------------------------------------
# 18. QWEN DOCUMENT ANSWER
# ------------------------------------------------------------

def generate_answer(
    question,
    context
):

    system_prompt = """
You are a document question-answering assistant.

Use ONLY the provided document context.

Rules:
- Do not invent facts.
- Do not use outside knowledge.
- Give a clear and simple answer.
- If the information is not present, say:
  "The answer is not available in the document."
"""

    user_prompt = f"""
DOCUMENT CONTEXT:
{context}

QUESTION:
{question}
"""

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt = (
        qwen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    )

    inputs = qwen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(DEVICE)

    with torch.no_grad():

        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False
        )

    new_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    answer = qwen_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


# ------------------------------------------------------------
# 19. COMPLETE DOCUMENT QA
# ------------------------------------------------------------

def answer_question(
    question,
    target_language
):

    global current_document_language

    if current_index is None:

        return (
            "Please upload and process a PDF first."
        )

    if not question or not question.strip():

        return (
            "Please enter or speak a question."
        )

    # Detect question language
    question_language = detect_language(
        question
    )

    # Convert question to English for the retrieval/LLM layer
    if question_language == "English":

        english_question = question

    elif question_language in LANG_CODES:

        english_question = translate_nllb(
            question,
            question_language,
            "English"
        )

    else:

        english_question = question

    # Search PDF
    results = search_document(
        english_question,
        top_k=3
    )

    if not results:

        return (
            "The answer is not available "
            "in the document."
        )

    context = "\n\n".join(
        r["chunk"]
        for r in results
    )

    # Convert context to English if needed
    if current_document_language in LANG_CODES:

        if current_document_language != "English":

            english_context = translate_nllb(
                context,
                current_document_language,
                "English"
            )

        else:

            english_context = context

    else:

        english_context = context

    # IndicBERT processing
    if indicbert_model is not None:

        # Generate representation of the question.
        # This is the Indian-language NLP module.
        _ = indicbert_encode(
            question
        )

    # Generate answer
    english_answer = generate_answer(
        english_question,
        english_context
    )

    # Return selected language
    if target_language == "English":

        return english_answer

    final_answer = translate_nllb(
        english_answer,
        "English",
        target_language
    )

    return final_answer


# ------------------------------------------------------------
# 20. FINAL USER QUESTION HANDLER
# ------------------------------------------------------------

def ask_ai(
    typed_question,
    audio_question,
    answer_language
):

    # Voice gets priority
    if audio_question is not None:

        question = transcribe_audio(
            audio_question
        )

    else:

        question = typed_question

    if not question or not question.strip():

        return (
            "",
            "Please type or speak a question."
        )

    answer = answer_question(
        question,
        answer_language
    )

    return (
        question,
        answer
    )


# ------------------------------------------------------------
# 21. GRADIO UI
# ------------------------------------------------------------

with gr.Blocks(
    title="Multilingual AI Document Assistant"
) as app:

    gr.Markdown(
        """
        # 🌐 Multilingual AI Document Assistant

        ### Upload a document → Ask by text or voice → Get an answer
        ### Supported answer languages: English • ಕನ್ನಡ • हिंदी
        """
    )

    # --------------------------------------------------------
    # DOCUMENT
    # --------------------------------------------------------

    gr.Markdown("## 📄 1. Upload Document")

    pdf_input = gr.File(
        label="Upload PDF",
        file_types=[".pdf"],
        type="filepath"
    )

    process_button = gr.Button(
        "🔍 Process Document",
        variant="primary"
    )

    document_status = gr.Textbox(
        label="Document Status",
        lines=7,
        interactive=False
    )

    # --------------------------------------------------------
    # QUESTION
    # --------------------------------------------------------

    gr.Markdown("## 💬 2. Ask a Question")

    typed_question = gr.Textbox(
        label="⌨️ Type your question",
        placeholder=(
            "Example: What are the main objectives "
            "of this project?"
        ),
        lines=4
    )

    audio_question = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="🎤 Or speak your question"
    )

    answer_language = gr.Radio(
        choices=[
            "English",
            "Kannada",
            "Hindi"
        ],
        value="English",
        label="🌐 Answer Language"
    )

    ask_button = gr.Button(
        "🤖 Ask AI",
        variant="primary"
    )

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    gr.Markdown("## 📝 3. Result")

    question_output = gr.Textbox(
        label="Question understood",
        lines=3,
        interactive=False
    )

    answer_output = gr.Textbox(
        label="🤖 AI Answer",
        lines=12,
        interactive=False
    )

    gr.Markdown(
        """
        ### Modules used

        📄 PyMuPDF → PDF text extraction
        🌐 LangDetect → language detection
        ✂️ Python → text chunking
        🧠 Sentence Transformer → multilingual embeddings
        🔎 FAISS → semantic document search
        🇮🇳 IndicBERT → Indian-language NLP representation
        🎤 Whisper → speech-to-text
        🤖 Qwen → document question answering
        🌍 NLLB → English / Kannada / Hindi translation
        🖥️ Gradio → user interface
        """
    )

    # --------------------------------------------------------
    # EVENTS
    # --------------------------------------------------------

    process_button.click(
        fn=process_pdf,
        inputs=pdf_input,
        outputs=document_status
    )

    ask_button.click(
        fn=ask_ai,
        inputs=[
            typed_question,
            audio_question,
            answer_language
        ],
        outputs=[
            question_output,
            answer_output
        ]
    )


# ------------------------------------------------------------
# 22. LAUNCH
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✅ ALL MODELS AND MODULES INITIALIZED")
print("=" * 70)

app.launch(
    share=True,
    debug=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have hu

🌐 MULTILINGUAL AI DOCUMENT ASSISTANT
Device: cpu
PyTorch: 2.11.0+cpu
✅ HF_TOKEN found in Colab Secrets

⏳ Loading multilingual embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cpu


✅ Whisper loaded

✅ ALL MODELS AND MODULES INITIALIZED
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6467cf48197ac71233.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENT